# DSA 4060: Personalized Movie Recommender Practical Quiz
- **Name:** ANGELA IRUNGU
- **Student ID:** 669289
- **Assigned User ID:** 10

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load datasets
movies_df = pd.read_csv('movies.csv')
ratings_df = pd.read_csv('ratings.csv')

# 1. Display first 5 rows
print("--- Movies (First 5 Rows) ---")
display(movies_df.head())

print("--- Ratings (First 5 Rows) ---")
display(ratings_df.head())

# 2. Display shape (rows, columns)
print(f"Movies dataset shape: {movies_df.shape}")
print(f"Ratings dataset shape: {ratings_df.shape}")

# 3. Check for missing values
print("\n--- Missing Values in Movies ---")
print(movies_df.isnull().sum())

print("\n--- Missing Values in Ratings ---")
print(ratings_df.isnull().sum())

--- Movies (First 5 Rows) ---


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


--- Ratings (First 5 Rows) ---


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


Movies dataset shape: (9742, 3)
Ratings dataset shape: (100836, 4)

--- Missing Values in Movies ---
movieId    0
title      0
genres     0
dtype: int64

--- Missing Values in Ratings ---
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


Both movies.csv and ratings.csv have zero missing values across all columns.

In [3]:
print("Movies columns:", movies_df.columns.tolist())
print("Ratings columns:", ratings_df.columns.tolist())

Movies columns: ['movieId', 'title', 'genres']
Ratings columns: ['userId', 'movieId', 'rating', 'timestamp']


In [4]:
ASSIGNED_USER_ID = 10

# Filter ratings for User 10 using 'userId' and merge using 'movieId'
user_ratings = ratings_df[ratings_df['userId'] == ASSIGNED_USER_ID]
user_profile_df = pd.merge(user_ratings, movies_df, on='movieId')

# 1. List all movies rated by User 10
print(f"--- All Movies Rated by User {ASSIGNED_USER_ID} ---")
display(user_profile_df[['movieId', 'title', 'genres', 'rating']])

# 2. Identify top 3 highest-rated movies
top_3_movies = user_profile_df.sort_values(by='rating', ascending=False).head(3)
print(f"\n--- Top 3 Highest-Rated Movies for User {ASSIGNED_USER_ID} ---")
display(top_3_movies[['title', 'genres', 'rating']])

# 3. Analyze genre preferences
user_profile_df['genre_list'] = user_profile_df['genres'].str.split('|')
genre_analysis = (
    user_profile_df.explode('genre_list')
    .groupby('genre_list')['rating']
    .agg(['count', 'mean'])
    .sort_values(by=['mean', 'count'], ascending=False)
)
print("\n--- Genre Preference Analysis ---")
display(genre_analysis)

--- All Movies Rated by User 10 ---


,movieId,title,genres,rating
0,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,1.0
1,356,Forrest Gump (1994),Comedy|Drama|Romance|War,3.5
2,588,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical,4.0
3,597,Pretty Woman (1990),Comedy|Romance,3.5
4,912,Casablanca (1942),Drama|Romance,4.0
...,...,...,...,...
135,119145,Kingsman: The Secret Service (2015),Action|Adventure|Comedy|Crime,1.0
136,129428,The Second Best Exotic Marigold Hotel (2015),Comedy|Drama,3.5
137,136020,Spectre (2015),Action|Adventure|Crime,5.0
138,137595,Magic Mike XXL (2015),Comedy|Drama,4.0



--- Top 3 Highest-Rated Movies for User 10 ---


,title,genres,rating
137,Spectre (2015),Action|Adventure|Crime,5.0
117,Skyfall (2012),Action|Adventure|Thriller|IMAX,5.0
110,"Dark Knight Rises, The (2012)",Action|Adventure|Crime|IMAX,5.0



--- Genre Preference Analysis ---


,count,mean
genre_list,,
Animation,15,3.866667
War,4,3.750000
Children,14,3.607143
Adventure,31,3.580645
Action,26,3.500000
Fantasy,17,3.441176
IMAX,18,3.361111
Romance,78,3.333333
Musical,9,3.333333


Based on User 10's rating history, their preferred genres are identified by evaluating highest average ratings alongside viewing frequency. High ratings (4.0+) are concentrated in specific genres, showing clear positive alignment with those themes.

In [6]:
# 1. Identify liked movies (rating >= 4.0)
liked_movies = user_profile_df[user_profile_df['rating'] >= 4.0]
liked_movie_ids = liked_movies['movieId'].tolist()
rated_movie_ids = user_ratings['movieId'].tolist()

# 2. Vectorize genres
movies_df['genres_processed'] = movies_df['genres'].str.replace('|', ' ', regex=False)
tfidf = TfidfVectorizer()
genre_matrix = tfidf.fit_transform(movies_df['genres_processed'])

# 3. Compute Cosine Similarity
cosine_sim = cosine_similarity(genre_matrix, genre_matrix)

movie_id_to_idx = {m_id: idx for idx, m_id in enumerate(movies_df['movieId'])}
liked_indices = [movie_id_to_idx[m_id] for m_id in liked_movie_ids if m_id in movie_id_to_idx]

# Average similarity across all liked movies
sim_scores = cosine_sim[liked_indices].mean(axis=0)
movies_df['sim_score'] = sim_scores

# 4. Filter out already rated movies and pick top 5
unseen_movies = movies_df[~movies_df['movieId'].isin(rated_movie_ids)]
top_5_recommendations = unseen_movies.sort_values(by='sim_score', ascending=False).head(5)

display(top_5_recommendations[['movieId', 'title', 'genres', 'sim_score']])

,movieId,title,genres,sim_score
5705,27808,Spanglish (2004),Comedy|Drama|Romance,0.505882
5711,27826,Touch of Pink (2004),Comedy|Drama|Romance,0.505882
47,52,Mighty Aphrodite (1995),Comedy|Drama|Romance,0.505882
7621,87234,Submarine (2010),Comedy|Drama|Romance,0.505882
4791,7137,"Cooler, The (2003)",Comedy|Drama|Romance,0.505882


In [7]:
recommendations_table = pd.DataFrame({
    'Rank': range(1, 6),
    'Recommended Movie': top_5_recommendations['title'].values,
    'Genres': top_5_recommendations['genres'].values,
    'Reason': [
        f"Matches genre profile of positively rated movies ({g})"
        for g in top_5_recommendations['genres'].values
    ]
})

display(recommendations_table)

,Rank,Recommended Movie,Genres,Reason
0,1,Spanglish (2004),Comedy|Drama|Romance,Matches genre profile of positively rated movi...
1,2,Touch of Pink (2004),Comedy|Drama|Romance,Matches genre profile of positively rated movi...
2,3,Mighty Aphrodite (1995),Comedy|Drama|Romance,Matches genre profile of positively rated movi...
3,4,Submarine (2010),Comedy|Drama|Romance,Matches genre profile of positively rated movi...
4,5,"Cooler, The (2003)",Comedy|Drama|Romance,Matches genre profile of positively rated movi...


### Task 5 - Critical Thinking

1. **Limitation of the Recommender:**
   - **Genre-Only Over-Specialization (Filter Bubble):** The current recommendation logic relies exclusively on movie genres. It cannot differentiate high-quality films from poorly rated ones within the same genre, nor does it factor in essential metadata such as directors, lead actors, plot keywords, or release year

2. **Realistic Improvement:**
   - **Hybrid Recommendation Approach:** Combine content-based filtering with collaborative filtering (e.g., Matrix Factorization / Singular Value Decomposition). Incorporating rating patterns from similar users alongside expanded movie metadata will improve personalization accuracy and increase recommendation diversity

**Recommendation Results**

The notebook generates a summary table of 5 recommended movies for User 10 along with rationale based on similarity to positively rated genre profiles.

**Limitation and Suggested Improvement**

**Limitation:** The recommender relies solely on genre tags, which can lead to over-specialization and ignore critical factors such as movie popularity, ratings quality, cast, or director.

**Improvement:** Implement a hybrid recommender model combining collaborative filtering (leveraging multi-user rating patterns) with richer content metadata to boost diversity and recommendation quality.